In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_patients = 200
n_genes = 100
biomarker_genes = ["GENE_5", "GENE_12", "GENE_27", "GENE_44", "GENE_88"]

# --------------------
# Clinical data
# --------------------
patients = pd.DataFrame({
    "patient_id": range(1, n_patients + 1),
    "age": np.random.normal(62, 8, n_patients).astype(int),
    "sex": np.random.choice(["M", "F"], n_patients)
})

# Base risk influenced by age
base_risk = (patients["age"] - 60) * 0.03

# --------------------
# Gene expression
# --------------------
genes = [f"GENE_{i}" for i in range(1, n_genes + 1)]

expression_data = []

risk_signal = np.zeros(n_patients)

for i, patient in patients.iterrows():
    patient_signal = 0

    for gene in genes:
        value = np.random.normal(5, 1)

        if gene in biomarker_genes:
            effect = np.random.normal(1.5, 0.3)
            value += effect
            patient_signal += effect

        expression_data.append([
            patient["patient_id"],
            gene,
            round(value, 4)
        ])

    risk_signal[i] = patient_signal

# Combine gene signal + age effect
risk_score = (base_risk * 0.5) + (risk_signal / 20)

# Add random noise
risk_score += np.random.normal(0, 0.5, n_patients)

# Convert to probability
prob = 1 / (1 + np.exp(-risk_score))

patients["risk_label"] = (prob > 0.5).astype(int)

expression = pd.DataFrame(expression_data, columns=["patient_id", "gene", "expression_value"])

patients.to_csv("patients.csv", index=False)
expression.to_csv("gene_expression.csv", index=False)

print("Dataset generated successfully.")

In [ ]:
print(patients.head())
print(expression.head())

In [ ]:
import sqlite3

conn = sqlite3.connect("clinical_genomics.db")
patients = pd.read_csv("patients.csv")
expression = pd.read_csv("gene_expression.csv")

patients.to_sql("patients", conn, if_exists="replace", index=False)
expression.to_sql("gene_expression", conn, if_exists="replace", index=False)
conn.commit()
conn.close()
print("Database created successfully.")

In [ ]:
query = """
SELECT
    g.gene,
    p.risk_label,
    AVG(g.expression_value) as mean_expression
FROM gene_expression g
JOIN patients p ON g.patient_id = p.patient_id
GROUP BY g.gene, p.risk_label
"""

In [ ]:
conn = sqlite3.connect("clinical_genomics.db")
result = pd.read_sql_query(query, conn)
print(result.head())

In [ ]:
pivot = result.pivot(index="gene", columns="risk_label", values="mean_expression")
pivot["difference"] = pivot[1] - pivot[0]
pivot.sort_values("difference", ascending=False).head(10)

In [ ]:
biomarker_genes = ["GENE_5", "GENE_12", "GENE_27", "GENE_44", "GENE_88"]

In [ ]:
pivot = result.pivot(index="gene", columns="risk_label", values="mean_expression")
pivot["difference"] = pivot[1] - pivot[0]
top_genes = pivot.sort_values("difference", ascending=False)
top_genes.head(10)

In [ ]:
top10 = top_genes.head(10)
print(top10.index.tolist())
set(biomarker_genes).intersection(set(top10.index))

In [ ]:
expression_wide = expression.pivot(
    index="patient_id",
    columns="gene",
    values="expression_value"
)
expression_wide.head()

In [ ]:
data = patients.merge(expression_wide, on="patient_id")
data.head()

In [ ]:
data["sex"] = data["sex"].map({"M": 0, "F": 1})

In [ ]:
X = data.drop(columns=["patient_id", "risk_label"])
y = data["risk_label"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", auc)

In [ ]:
coef_df = pd.DataFrame({"feature": X.columns, "coefficient": model.coef_[0]})
coef_df.sort_values("coefficient", ascending=False).head(10)

In [ ]:
# Exportar dataset final com probabilidades
X_aligned = X.copy()
data["predicted_probability"] = model.predict_proba(X_aligned)[:, 1]
data.to_csv("final_clinical_genomic_dataset.csv", index=False)
print("Dataset final exportado com sucesso.")
print("Shape:", data.shape)
print(data.head())